# EGN 321 — Module 8 Assignment 8.1 Complete Instructor Demo
## The Silent Handoff Challenge: Documentation, Operation, and Improvement

This notebook demonstrates the complete Assignment 8.1 experience using a realistic application and **900 sensor-history records**.

### Demo application
**Cooling Loop Reliability Monitor**

The notebook demonstrates:
- realistic application records;
- validation and refusal behavior;
- SQLite history;
- operational logs;
- Flask dashboard verification;
- README quality review;
- the 12 required silent-handoff tasks;
- handoff observation evidence;
- scoring how far the operator gets without help;
- identifying documentation failures;
- making at least two meaningful improvements;
- optional Ollama documentation review;
- exporting a complete Assignment 8.1 evidence package.


## Central Question

> **Can somebody else successfully use this system without the person who created it?**

```text
Repository
   ↓
Documentation
   ↓
Installation
   ↓
Application Startup
   ↓
Normal Operation
   ↓
Interpretation
   ↓
Logs / Health
   ↓
Troubleshooting
   ↓
Recovery
   ↓
Known Limitations
```


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import sqlite3
import json
import shutil
import zipfile
import subprocess
import sys
import os

DEMO_ROOT = Path("/content/module8_assignment8_1_demo")
DEMO_ROOT.mkdir(exist_ok=True)

print("Demo root:", DEMO_ROOT)


# Part A — Upload the 900-Record Dataset

Upload:

`cooling_loop_sensor_history_900_records.csv`


In [ ]:
from google.colab import files

uploaded = files.upload()

filename = "cooling_loop_sensor_history_900_records.csv"

if filename not in uploaded:
    raise FileNotFoundError(
        f"Upload {filename}"
    )

DATA_FILE = DEMO_ROOT / filename
DATA_FILE.write_bytes(uploaded[filename])

sensor_df = pd.read_csv(DATA_FILE)
sensor_df["timestamp"] = pd.to_datetime(
    sensor_df["timestamp"],
    utc=True
)

print("Records:", len(sensor_df))
print("Columns:", list(sensor_df.columns))
display(sensor_df.head())


## Reality Check

The dataset intentionally contains:
- normal operation;
- sustained high temperature;
- missing flow;
- impossible temperatures;
- a communications gap;
- pressure out of range;
- sustained low flow;
- recovery.


In [ ]:
display(
    sensor_df["event_label"]
    .value_counts()
    .rename_axis("event")
    .reset_index(name="count")
)

display(
    sensor_df["source_status"]
    .value_counts()
    .rename_axis("source_status")
    .reset_index(name="count")
)


# Part B — Build Application Evidence


In [ ]:
def validate_reading(temp_c, pressure_kpa, flow_lpm):
    reasons = []

    if pd.isna(temp_c):
        reasons.append("temperature missing")
    elif not 0 <= float(temp_c) <= 60:
        reasons.append(f"temperature out of range: {temp_c}")

    if pd.isna(pressure_kpa):
        reasons.append("pressure missing")
    elif not 50 <= float(pressure_kpa) <= 400:
        reasons.append(f"pressure out of range: {pressure_kpa}")

    if pd.isna(flow_lpm):
        reasons.append("flow missing")
    elif not 0 <= float(flow_lpm) <= 200:
        reasons.append(f"flow out of range: {flow_lpm}")

    return len(reasons) == 0, reasons


class AlertEngine:
    def __init__(self):
        self.temp_count = 0
        self.flow_count = 0
        self.high_temp = False
        self.low_flow = False

    def update(self, temp_c, flow_lpm, accepted):
        if not accepted:
            self.temp_count = 0
            self.flow_count = 0
            return self.state()

        if not self.high_temp:
            self.temp_count = self.temp_count + 1 if temp_c >= 30.0 else 0
            if self.temp_count >= 3:
                self.high_temp = True
        elif temp_c <= 28.5:
            self.high_temp = False
            self.temp_count = 0

        if not self.low_flow:
            self.flow_count = self.flow_count + 1 if flow_lpm < 25.0 else 0
            if self.flow_count >= 3:
                self.low_flow = True
        elif flow_lpm >= 35.0:
            self.low_flow = False
            self.flow_count = 0

        return self.state()

    def state(self):
        alerts = []
        if self.high_temp:
            alerts.append("HIGH_TEMPERATURE")
        if self.low_flow:
            alerts.append("LOW_FLOW")
        return alerts or ["NORMAL"]


In [ ]:
DB_FILE = DEMO_ROOT / "cooling_loop.db"
LOG_FILE = DEMO_ROOT / "system_log.jsonl"

for p in [DB_FILE, LOG_FILE]:
    if p.exists():
        p.unlink()

conn = sqlite3.connect(DB_FILE)

conn.execute('''
CREATE TABLE readings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    record_id INTEGER,
    timestamp TEXT,
    temperature_c REAL,
    pressure_kpa REAL,
    flow_lpm REAL,
    accepted INTEGER,
    health_state TEXT,
    alert_state TEXT,
    reason TEXT,
    gap_seconds REAL
)
''')

engine = AlertEngine()
previous_ts = None

for _, row in sensor_df.iterrows():
    ts = row["timestamp"]
    gap_seconds = 0 if previous_ts is None else (ts - previous_ts).total_seconds()
    previous_ts = ts

    accepted, reasons = validate_reading(
        row["temperature_c"],
        row["pressure_kpa"],
        row["flow_lpm"]
    )

    health = "HEALTHY"
    if not accepted or gap_seconds > 30:
        health = "DEGRADED"

    temp = None if pd.isna(row["temperature_c"]) else float(row["temperature_c"])
    flow = None if pd.isna(row["flow_lpm"]) else float(row["flow_lpm"])

    alerts = engine.update(temp, flow, accepted)
    reason = "; ".join(reasons)

    conn.execute(
        '''INSERT INTO readings
        (record_id,timestamp,temperature_c,pressure_kpa,flow_lpm,
         accepted,health_state,alert_state,reason,gap_seconds)
        VALUES(?,?,?,?,?,?,?,?,?,?)''',
        (
            int(row["record_id"]),
            ts.isoformat(),
            None if pd.isna(row["temperature_c"]) else float(row["temperature_c"]),
            None if pd.isna(row["pressure_kpa"]) else float(row["pressure_kpa"]),
            None if pd.isna(row["flow_lpm"]) else float(row["flow_lpm"]),
            int(accepted),
            health,
            ",".join(alerts),
            reason,
            gap_seconds
        )
    )

    log_record = {
        "timestamp": ts.isoformat(),
        "record_id": int(row["record_id"]),
        "component": "ingestion",
        "event": "reading_processed",
        "accepted": bool(accepted),
        "health_state": health,
        "alert_state": alerts,
        "reason": reason,
        "gap_seconds": gap_seconds
    }

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(log_record) + "\n")

conn.commit()
history = pd.read_sql_query(
    "SELECT * FROM readings ORDER BY id",
    conn
)
conn.close()

print("Database rows:", len(history))
display(history.head())


In [ ]:
print("Accepted / rejected:")
display(
    history["accepted"]
    .value_counts()
    .rename_axis("accepted")
    .reset_index(name="count")
)

print("Health states:")
display(
    history["health_state"]
    .value_counts()
    .rename_axis("health_state")
    .reset_index(name="count")
)

print("Alert states:")
display(
    history["alert_state"]
    .value_counts()
    .rename_axis("alert_state")
    .reset_index(name="count")
)

print("Degraded or rejected examples:")
display(
    history[
        (history["accepted"] == 0) |
        (history["health_state"] != "HEALTHY")
    ].head(20)
)


# Part C — Verify a Flask Handoff Application

Colab is used as the teaching environment. Flask's test client verifies the application without pretending Colab is a permanent deployment.


In [ ]:
%pip install -q flask pytest

APP_DIR = DEMO_ROOT / "handoff_app"
APP_DIR.mkdir(exist_ok=True)

shutil.copy2(DB_FILE, APP_DIR / "cooling_loop.db")
shutil.copy2(LOG_FILE, APP_DIR / "system_log.jsonl")
shutil.copy2(DATA_FILE, APP_DIR / DATA_FILE.name)

app_code = '''
from flask import Flask, jsonify, render_template_string
import sqlite3
from pathlib import Path

BASE = Path(__file__).parent
DB = BASE / "cooling_loop.db"

app = Flask(__name__)

PAGE = """
<!doctype html>
<html>
<head>
<title>Cooling Loop Reliability Monitor</title>
<style>
body { font-family: Arial, sans-serif; margin: 36px; }
.cards { display:grid; grid-template-columns:repeat(4,1fr); gap:12px; }
.card { border:1px solid #ccc; border-radius:8px; padding:15px; }
table { width:100%; border-collapse:collapse; margin-top:24px; }
th,td { border:1px solid #ddd; padding:7px; }
</style>
</head>
<body>
<h1>Cooling Loop Reliability Monitor</h1>
<p>Module 8 Silent Handoff Demo</p>
<div class="cards">
<div class="card"><b>Latest Record</b><br>{{ latest[0] }}</div>
<div class="card"><b>Health</b><br>{{ latest[6] }}</div>
<div class="card"><b>Alert</b><br>{{ latest[7] }}</div>
<div class="card"><b>Total Records</b><br>{{ count }}</div>
</div>
<table>
<tr>
<th>Record</th><th>Timestamp</th><th>Temp C</th><th>Pressure kPa</th>
<th>Flow L/min</th><th>Accepted</th><th>Health</th><th>Alert</th>
</tr>
{% for row in recent %}
<tr>
{% for value in row %}
<td>{{ value }}</td>
{% endfor %}
</tr>
{% endfor %}
</table>
</body>
</html>
"""

@app.route("/")
def dashboard():
    conn = sqlite3.connect(DB)
    latest = conn.execute(
        """SELECT record_id,timestamp,temperature_c,pressure_kpa,flow_lpm,
                  accepted,health_state,alert_state
           FROM readings ORDER BY id DESC LIMIT 1"""
    ).fetchone()

    recent = conn.execute(
        """SELECT record_id,timestamp,temperature_c,pressure_kpa,flow_lpm,
                  accepted,health_state,alert_state
           FROM readings ORDER BY id DESC LIMIT 12"""
    ).fetchall()

    count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
    conn.close()

    return render_template_string(
        PAGE,
        latest=latest,
        recent=recent,
        count=count
    )

@app.route("/api/status")
def api_status():
    conn = sqlite3.connect(DB)
    row = conn.execute(
        """SELECT record_id,timestamp,health_state,alert_state,reason
           FROM readings ORDER BY id DESC LIMIT 1"""
    ).fetchone()
    conn.close()

    return jsonify({
        "record_id": row[0],
        "timestamp": row[1],
        "health_state": row[2],
        "alert_state": row[3],
        "reason": row[4]
    })

if __name__ == "__main__":
    app.run(port=5000)
'''

(APP_DIR / "app.py").write_text(
    app_code,
    encoding="utf-8"
)

(APP_DIR / "requirements.txt").write_text(
    "flask\npandas\npytest\n",
    encoding="utf-8"
)


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "handoff_app",
    APP_DIR / "app.py"
)

app_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_module)

client = app_module.app.test_client()

dashboard_response = client.get("/")
status_response = client.get("/api/status")

print("Dashboard status:", dashboard_response.status_code)
print("Status endpoint:", status_response.status_code)
print(json.dumps(status_response.get_json(), indent=2))

assert dashboard_response.status_code == 200
assert status_response.status_code == 200


# Part D — Documentation Quality: Before vs After


In [ ]:
README_BEFORE = '''# Cooling Loop Reliability Monitor

Install the dependencies and run the application.

Use the included data.

If something goes wrong, check the logs.
'''

README_FINAL = '''# Cooling Loop Reliability Monitor

## Project Overview
This application monitors cooling-loop temperature, pressure, and flow.

## Requirements
Python 3.10+, Flask, pandas, pytest, browser, local SQLite write access.

## Quick Start
1. Install requirements.
2. Build the database.
3. Start the Flask app.
4. Open http://127.0.0.1:5000.
5. Confirm 900 records appear.

## Inputs and Units
Temperature: Celsius, 0-60 C.
Pressure: kPa, 50-400 kPa.
Flow: L/min, 0-200 L/min.

## What the Tool Refuses
Missing inputs and values outside supported ranges.

## Logs
system_log.jsonl

## Recovery
If the database is missing, rebuild it, restart the app,
and verify that the dashboard reports 900 records.

## Limitation
The classroom demo does not independently verify physical sensor calibration.
'''

print("BEFORE:\n")
print(README_BEFORE)

print("\n" + "="*60 + "\n")

print("AFTER:\n")
print(README_FINAL)


## Instructor Prompt

Ask:

> If I hand you the BEFORE version and leave the room, what questions would you have?

Then:

> What uncertainty did the AFTER version remove?


# Part E — Silent Handoff Task Sheet


In [ ]:
handoff_tasks = pd.DataFrame([
    [1, "Locate the application and repository files"],
    [2, "Explain what the system is designed to do"],
    [3, "Install or verify required dependencies"],
    [4, "Start the application"],
    [5, "Complete one normal workflow"],
    [6, "Provide or identify valid input"],
    [7, "Interpret the displayed result"],
    [8, "Locate health/status information"],
    [9, "Locate the logs"],
    [10, "Identify one documented failure condition"],
    [11, "Follow one troubleshooting/recovery procedure"],
    [12, "Identify one known limitation"]
], columns=["step","handoff_task"])

display(handoff_tasks)


# Part F — Example Handoff Observation

In the real assignment, students replace this sample with observations from their assigned partner.


In [ ]:
handoff_observation = pd.DataFrame([
    [1,"Locate application","Yes","Found app.py and README quickly","Repository view","None"],
    [2,"Understand purpose","Yes","Purpose clear","README overview","None"],
    [3,"Verify dependencies","Yes","requirements.txt matched README","requirements.txt","None"],
    [4,"Start application","Yes","Startup command clear","Terminal output","None"],
    [5,"Normal workflow","Yes","Dashboard opened and showed 900 records","Dashboard","None"],
    [6,"Valid input","Yes","Included dataset was clearly identified","data file","None"],
    [7,"Interpret result","Partial","Alert meanings required rereading","README section 9","Add concrete alert examples"],
    [8,"Locate health/status","Yes","Health shown on dashboard/API","Dashboard/API","None"],
    [9,"Locate logs","Yes","Log path documented","system_log.jsonl","None"],
    [10,"Failure condition","Yes","Missing DB scenario documented","README troubleshooting","None"],
    [11,"Recovery procedure","Yes","Rebuilt DB and verified 900 rows","Recovery output","None"],
    [12,"Known limitation","Yes","Found limitation section","README limitations","None"]
], columns=[
    "step","operator_action","success",
    "confusion_observed","evidence","improvement_needed"
])

display(handoff_observation)


## Quick Handoff Level

This is a teaching metric only; it does not replace the rubric.


In [ ]:
score_map = {
    "Yes": 1.0,
    "Partial": 0.5,
    "No": 0.0
}

handoff_observation["score"] = (
    handoff_observation["success"]
    .map(score_map)
)

score = handoff_observation["score"].sum()
maximum = len(handoff_observation)
percent = score / maximum * 100

if percent >= 95:
    level = "Level 5 - Professional Handoff"
elif percent >= 85:
    level = "Level 4 - Transferable"
elif percent >= 70:
    level = "Level 3 - Operational"
elif percent >= 50:
    level = "Level 2 - Almost Operational"
else:
    level = "Level 1 - Needs the Developer"

print(f"Handoff score: {score:.1f}/{maximum}")
print(f"Percent: {percent:.1f}%")
print(level)


# Part G — Improve the System

Students must make at least two meaningful improvements.


In [ ]:
improvements = pd.DataFrame([
    {
        "problem_discovered":
            "Operator understood that alerts existed but had to reread the guide to interpret them.",
        "change_made":
            "Add an alert-reference table showing NORMAL, HIGH_TEMPERATURE, and LOW_FLOW with examples.",
        "why_it_helps":
            "The operator can interpret a dashboard state without searching through prose."
    },
    {
        "problem_discovered":
            "Recovery instructions explained what to run but did not define a clear success check.",
        "change_made":
            "Add explicit verification: dashboard must show 900 records and /api/status must return a valid state.",
        "why_it_helps":
            "Recovery becomes evidence-based instead of relying on 'it looks like it started.'"
    }
])

display(improvements)


# Part H — Required Reflection

1. What did your partner struggle with that seemed obvious to you?
2. What was the most important improvement made after the handoff?
3. If you could no longer support the application personally, what additional documentation or automation would you add?


# Part I — Optional Ollama Documentation Review

Ollama can act as a first-time-user reviewer, but it does **not** replace the human silent handoff.


In [ ]:
OLLAMA_REVIEW_PROMPT = '''
You are reviewing operating documentation as a first-time user.

Do NOT rewrite the guide yet.

Identify:
1. missing prerequisites;
2. ambiguous setup steps;
3. instructions that assume prior knowledge;
4. unclear success checks;
5. missing troubleshooting or recovery information;
6. missing assumptions or limitations.

For every issue:
- identify the relevant section;
- explain why a first-time operator may struggle;
- recommend one concrete improvement.

Do not invent application features.
'''

print(OLLAMA_REVIEW_PROMPT)


# Part J — Export Assignment 8.1 Evidence


In [ ]:
EXPORT = DEMO_ROOT / "assignment8_1_export"
EXPORT.mkdir(exist_ok=True)

handoff_tasks.to_csv(
    EXPORT / "handoff_task_list.csv",
    index=False
)

handoff_observation.to_csv(
    EXPORT / "handoff_observation.csv",
    index=False
)

improvements.to_csv(
    EXPORT / "improvements_after_handoff.csv",
    index=False
)

history.to_csv(
    EXPORT / "application_history_900_records.csv",
    index=False
)

summary = {
    "records_processed": int(len(history)),
    "accepted_records": int((history["accepted"] == 1).sum()),
    "rejected_records": int((history["accepted"] == 0).sum()),
    "degraded_records": int((history["health_state"] == "DEGRADED").sum()),
    "handoff_score": float(score),
    "handoff_maximum": int(maximum),
    "handoff_percent": float(percent),
    "handoff_level": level
}

(EXPORT / "handoff_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

(EXPORT / "reflection_template.md").write_text(
'''# Assignment 8.1 Reflection

## 1. What did your partner struggle with that seemed obvious to you?

## 2. What was the most important improvement you made after the handoff?

## 3. If you could no longer support this application personally, what additional documentation or automation would you add?
''',
    encoding="utf-8"
)

(EXPORT / "AI_LOG.md").write_text(
'''# AI_LOG.md

## Tool / Model

## Task

## Prompt

## AI Output Used

## AI Output Rejected or Modified

## Verification Performed
''',
    encoding="utf-8"
)

print(json.dumps(summary, indent=2))


In [ ]:
ZIP_FILE = Path(
    "/content/EGN321_Module8_Assignment8_1_DEMO_EVIDENCE.zip"
)

if ZIP_FILE.exists():
    ZIP_FILE.unlink()

with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT.iterdir():
        if p.is_file():
            z.write(p, arcname=p.name)

print("Created:", ZIP_FILE)

for p in sorted(EXPORT.iterdir()):
    print(" -", p.name)

# Optional:
# from google.colab import files
# files.download(str(ZIP_FILE))


# Instructor Closing Discussion

Ask:

> What failed first: the software, or the instructions?

Then:

> If the developer disappeared tomorrow, what would the next operator still need?

**Documentation is part of the application.  
Handoff is a test.  
Confusion is evidence.  
Improvement is the response.**
